# 🚀 HUẤN LUYỆN & THỰC NGHIỆM MỞ RỘNG SEQ2SEQ LSTM + ATTENTION
**Đề tài 8: Dịch máy Anh - Việt | Môn học: Học sâu**  
**Phụ trách: Hoàng (Phần 2) • Chạy train & W&B: Sơn (Phần 3)**  
---
> **Lưu ý:** Chọn GPU trên Colab: **Runtime -> Change runtime type -> T4 GPU**.

### Bước 1: Kiểm tra GPU & Kết nối Google Drive chung

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Clone / Cập nhật mã nguồn từ nhánh `main` & Cài đặt thư viện

In [ ]:
import os

REPO_URL = "https://github.com/hoang180/Machine_Translation_Project.git"
PROJECT_DIR = "/content/Machine_Translation_Project"
BRANCH = "main"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("✓ Môi trường và mã nguồn nhánh main đã sẵn sàng!")

### Bước 3: Nạp dữ liệu `data/processed/` & 3 bộ Tokenizer từ Google Drive chung (`BTL_DL_Translation`)

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"

!mkdir -p data/processed custom_tokenizers/word custom_tokenizers/bpe custom_tokenizers/spm checkpoints outputs
!cp -r "{DRIVE_ROOT}/data/processed/." data/processed/
!cp -r "{DRIVE_ROOT}/custom_tokenizers/." custom_tokenizers/

!ls -lh data/processed/ custom_tokenizers/word/ custom_tokenizers/bpe/ custom_tokenizers/spm/

### Bước 4: Đăng nhập Weights & Biases (`btl-dl-translation-iwslt15`)

In [ ]:
import wandb
wandb.login()

### Bước 5: Huấn luyện Baseline & các cấu hình mở rộng Seq2seq
Chạy lần lượt (ưu tiên chạy **BPE + Luong** trước để chốt Baseline cùng Transformer):
1. **Baseline BPE + Luong** (`seq2seq-luong-bpe`)
2. **Word + Luong** (`seq2seq-luong-word`)
3. **SentencePiece + Luong** (`seq2seq-luong-spm`)
4. **BPE + Bahdanau** (`seq2seq-bahdanau-bpe`)

In [ ]:
# 1. Ưu tiên chạy trước: BPE + Luong (Baseline)
!python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer bpe --attention luong --save_dir checkpoints/seq2seq_bpe_luong --batch_size 128 --epochs 15

# Sao lưu ngay checkpoint vừa train lên Drive
!mkdir -p "{DRIVE_ROOT}/checkpoints/seq2seq_bpe_luong"
!cp -r checkpoints/seq2seq_bpe_luong/. "{DRIVE_ROOT}/checkpoints/seq2seq_bpe_luong/"

In [ ]:
# --- Các cấu hình mở rộng Giai đoạn 3 (bỏ dấu # khi chạy) ---

# 2. Word + Luong
# !python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer word --attention luong --save_dir checkpoints/seq2seq_word_luong --batch_size 128 --epochs 15

# 3. SentencePiece + Luong
# !python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer spm --attention luong --save_dir checkpoints/seq2seq_spm_luong --batch_size 128 --epochs 15

# 4. BPE + Bahdanau
# !python3 train_seq2seq.py --config configs/seq2seq_config.yaml --tokenizer bpe --attention bahdanau --save_dir checkpoints/seq2seq_bpe_bahdanau --batch_size 128 --epochs 15

### Bước 6: Đánh giá SacreBLEU (`eval.py`), Benchmark Decoding & Phân tích lỗi (`tst2013`)

In [ ]:
# Đánh giá nhanh Greedy (k=1) & Beam Search (k=4)
!python3 eval.py --checkpoint checkpoints/seq2seq_bpe_luong/best_model.pt --beam_size 1
!python3 eval.py --checkpoint checkpoints/seq2seq_bpe_luong/best_model.pt --beam_size 4 --alpha 0.6

# Benchmark Decoding (Greedy vs Beam k=3,5) + Phân tích lỗi theo độ dài (<15, 15-30, >30 từ)
!python3 error_analysis.py --checkpoint checkpoints/seq2seq_bpe_luong/best_model.pt --benchmark_decoding --out_dir outputs/error_analysis_seq2seq_bpe

### Bước 7: Demo dịch thử & Sao lưu toàn bộ Checkpoints / Kết quả lên Google Drive

In [ ]:
!python3 demo.py --seq2seq_ckpt checkpoints/seq2seq_bpe_luong/best_model.pt --text "When I was little, I thought my country was the best on the planet."

!mkdir -p "{DRIVE_ROOT}/checkpoints" "{DRIVE_ROOT}/results"
!cp -r checkpoints/. "{DRIVE_ROOT}/checkpoints/"
!cp -r outputs/. "{DRIVE_ROOT}/results/"
print("✓ Đã sao lưu toàn bộ checkpoints và bảng kết quả lên Google Drive!")